In [2]:
#!pip install openai

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import openai
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
openai.api_key = '****************************************'

In [5]:
df = pd.read_excel('/content/hec-sim.xlsx')
df.head()

,Id,Full,Group,class,invCnt,greens,reds,controv,winning
0,39,Improve Elective Selection and Choice Process ...,A,highPriority,10,10,3,0.488770,NaN
1,40,Assign alumni mentors : All HEC Paris MBA stud...,A,neutral,11,9,3,0.698242,NaN
2,43,Improve access to Alumni network incl. Dynamic...,A,highPriority,15,14,2,0.031128,NaN
3,49,"Enhance professor diversity (gender, sectors) ...",A,controversial,21,14,10,2.922499,NaN
4,52,Access to past student reviews on elective mod...,A,highPriority,18,14,5,0.443573,1.0


In [6]:
## Generate Embeddings
def get_embedding(text, model="text-embedding-ada-002"):
    text = text.replace("\n", " ")
    response = openai.embeddings.create(
        model=model,
        input=[text]
    )
    return response.data[0].embedding

# Generate embeddings for all ideas
df['embedding'] = df['Full'].apply(get_embedding)

In [8]:
df.head(2)
df.to_csv('ada_embeddings.csv', index=False)

In [9]:
## Calculate Similarities
def cosine_sim(emb1, emb2):
    return cosine_similarity([emb1], [emb2])[0][0]

# Calculate similarity matrix
similarity_matrix = cosine_similarity(df['embedding'].tolist())

In [10]:
# Find the most similar idea in the other group for each idea
df['most_similar_other_group'] = df.apply(lambda row: df[df['Group'] != row['Group']]['Id'].iloc[np.argmax(similarity_matrix[row.name][df['Group'] != row['Group']])], axis=1)
df['most_similar_own_group'] = df.apply(lambda row: df[(df['Group'] == row['Group']) & (df.index != row.name)]['Id'].iloc[np.argmax(similarity_matrix[row.name][(df['Group'] == row['Group']) & (df.index != row.name)])], axis=1)
df['similarity_other_group'] = df.apply(lambda row: np.max(similarity_matrix[row.name][df['Group'] != row['Group']]), axis=1)
df['similarity_own_group'] = df.apply(lambda row: np.max(similarity_matrix[row.name][(df['Group'] == row['Group']) & (df.index != row.name)]), axis=1)

In [11]:
df.head(3)

,Id,Full,Group,class,invCnt,greens,reds,controv,winning,embedding,most_similar_other_group,most_similar_own_group,similarity_other_group,similarity_own_group
0,39,Improve Elective Selection and Choice Process ...,A,highPriority,10,10,3,0.488770,NaN,"[-0.013399704359471798, -0.0056852358393371105...",98,44,0.923561,0.871704
1,40,Assign alumni mentors : All HEC Paris MBA stud...,A,neutral,11,9,3,0.698242,NaN,"[-0.007057524751871824, -0.023907283321022987,...",18,32,0.849015,0.852988
2,43,Improve access to Alumni network incl. Dynamic...,A,highPriority,15,14,2,0.031128,NaN,"[-0.02285899966955185, -0.012485047802329063, ...",105,111,0.898296,0.870480


In [ ]:
#df = df.drop(columns=['embedding'])
#df.to_csv("/content/hec-sim.csv")

In [ ]:
#df.to_excel("/content/hec-sim.xlsx", index=False)